# Sentiment Analysis Engine — Evaluation & Analysis

End-to-end evaluation of the fine-tuned model: classification report, confusion matrix, per-class F1 (before vs after augmentation), error analysis, latency and throughput benchmarks.

Run `python -m src.trainer` first so a trained model exists in `MODEL_DIR`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_from_disk
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from config import get_settings, ID2LABEL
from src.inference import SentimentInferenceEngine

settings = get_settings()
settings.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print('Model dir:', settings.MODEL_DIR)

## 1. Load the test dataset and run full inference

In [ ]:
splits = load_from_disk(str(settings.PROCESSED_DIR))
test_ds = splits['test']
texts = test_ds['text']
y_true = [int(l) for l in test_ds['label']]

engine = SentimentInferenceEngine(model_dir=settings.MODEL_DIR, device='auto')
results = engine.predict_batch(texts, batch_size=32)
label2id = {v: k for k, v in ID2LABEL.items()}
y_pred = [label2id[r.label] for r in results]
print('Scored', len(y_pred), 'samples with backend', engine.backend)

## 2. Classification report

In [ ]:
target_names = [ID2LABEL[i] for i in sorted(ID2LABEL)]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

## 3. Confusion matrix heatmap

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=sorted(ID2LABEL))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.tight_layout()
out = settings.REPORTS_DIR / 'confusion_matrix.png'
plt.savefig(out, dpi=150)
print('Saved', out)
plt.show()

## 4. Per-class F1: before vs after augmentation

Replace the placeholder `f1_before` values with the metrics from a baseline model trained without augmentation.

In [ ]:
f1_after = f1_score(y_true, y_pred, average=None, labels=sorted(ID2LABEL), zero_division=0)
# Placeholder baseline (no augmentation). Update with real baseline metrics.
f1_before = np.array([0.70, 0.55, 0.78])

x = np.arange(len(target_names)); width = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, f1_before, width, label='Before augmentation')
ax.bar(x + width/2, f1_after, width, label='After augmentation')
ax.set_xticks(x); ax.set_xticklabels(target_names)
ax.set_ylabel('F1 score'); ax.set_title('Per-class F1: before vs after augmentation')
ax.legend(); plt.tight_layout()
plt.savefig(settings.REPORTS_DIR / 'f1_before_after.png', dpi=150)
plt.show()

## 5. Error analysis — 20 misclassified samples

In [ ]:
rows = []
for text, yt, r in zip(texts, y_true, results):
    yp = label2id[r.label]
    if yp != yt:
        rows.append({
            'text': text[:120],
            'true_label': ID2LABEL[yt],
            'predicted_label': r.label,
            'confidence': round(r.confidence, 4),
        })
errors_df = pd.DataFrame(rows).head(20)
errors_df

## 6. Latency benchmark — 1000 single predictions

In [ ]:
sample_texts = (texts * ((1000 // max(len(texts), 1)) + 1))[:1000]
latencies = []
for t in sample_texts:
    start = time.perf_counter()
    engine.predict(t)
    latencies.append((time.perf_counter() - start) * 1000.0)
latencies = np.array(latencies)
p50, p95, p99 = np.percentile(latencies, [50, 95, 99])

plt.figure(figsize=(8, 5))
plt.hist(latencies, bins=50, color='steelblue', alpha=0.8)
for val, label, color in [(p50, 'p50', 'green'), (p95, 'p95', 'orange'), (p99, 'p99', 'red')]:
    plt.axvline(val, color=color, linestyle='--', label=f'{label} = {val:.1f} ms')
plt.xlabel('Latency (ms)'); plt.ylabel('Count'); plt.title('Single-prediction latency')
plt.legend(); plt.tight_layout()
plt.savefig(settings.REPORTS_DIR / 'latency_hist.png', dpi=150)
plt.show()
print(f'p50={p50:.1f}ms  p95={p95:.1f}ms  p99={p99:.1f}ms')

## 7. Throughput test — batch sizes 1, 8, 16, 32, 64

In [ ]:
bench_texts = (texts * ((512 // max(len(texts), 1)) + 1))[:512]
throughput = {}
for bs in [1, 8, 16, 32, 64]:
    start = time.perf_counter()
    engine.predict_batch(bench_texts, batch_size=bs)
    elapsed = time.perf_counter() - start
    throughput[bs] = len(bench_texts) / elapsed

plt.figure(figsize=(7, 5))
plt.plot(list(throughput.keys()), list(throughput.values()), marker='o')
plt.xlabel('Batch size'); plt.ylabel('Predictions / second')
plt.title('Throughput vs batch size'); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(settings.REPORTS_DIR / 'throughput.png', dpi=150)
plt.show()
for bs, tps in throughput.items():
    print(f'batch_size={bs:>2}: {tps:8.1f} predictions/sec')